# Four Agentic Patterns

In this notebook, we look at four agentic patterns: (1) **reflection**, (2) **tool use**, (3) **planning** / **ReAct** [@ReAct2023], (4) **multi-agent**. This notebook builds on [this blog series](https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io) by [`@deeplearning.ai`](https://www.deeplearning.ai/) and the course by [`@neural-maze`](https://github.com/neural-maze/agentic-patterns-course?tab=readme-ov-file). These are summarized in the following table:



| Pattern              | Key Components                              | Description                                                                 |
|----------------------|---------------------------------------------|-----------------------------------------------------------|
| **Reflection**       | Generate ↔ Reflect                          | Iterative cycle where the model produces outputs, reflects on them, and improves future generations. |
| **Tool Use**         | Multiple tools to external systems | The model selects and applies external tools to extend its capabilities.   |
| **Planning (ReAct)** | Thought ↔ Action ↔ Observation              | Combines reasoning ("thought") with actions and feedback from observations in a loop. |
| **Multi-Agent**      | Agent 1 → Agent 2 → Agent 3       | Multiple agents collaborate or sequence their actions to solve complex tasks. |



## Utility functions

### Chat completion

Usual boilerplate when dealing with chat completions.

In [118]:
class Role:
    USER = "user"
    TOOL = "tool"
    SYSTEM = "system"
    ASSISTANT = "assistant"
    
    @classmethod
    def validate(cls, role: str) -> str:
        valid_roles = {cls.SYSTEM, cls.USER, cls.ASSISTANT}
        if role not in valid_roles:
            raise ValueError(f"Invalid role: {role}")
        return role


def completions_create(client, messages: list, model: str) -> str:
    """Return generated string from model based on messages."""
    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def message_dict(prompt: str, role: str, tag: str = "") -> dict:
    """Return a message dictionary for the chat completions API."""
    Role.validate(role)
    prompt = f"<{tag}>{prompt}</{tag}>" if tag else prompt
    return {"role": role, "content": prompt}


# example
try:
    print(message_dict(role="user", prompt="Hello", tag="greeting"))
    print(message_dict(role="test", prompt="Hello", tag="greeting"))
except ValueError as e:
    print(e)

{'role': 'user', 'content': '<greeting>Hello</greeting>'}
Invalid role: test


Implementing chat history class to abstract appending messages with limit to naively prevent ["context overflow"](https://aws.amazon.com/blogs/security/context-window-overflow-breaking-the-barrier/). We have the parameter `fixed_n` (default `1`) to preserve first `n` message since it is often important (e.g. `n=1` for the system prompt).

In [122]:
from typing import Optional


class ChatHistory(list):
    def __init__(self, 
        messages: Optional[list] = None, 
        system_prompt: Optional[str] = None, 
        max_len: int = -1, 
        fixed_n: int = 1
    ):
        """Fixed message list with a optional total length and number of fixed initial messages."""
        messages = [] if messages is None else messages
        super().__init__(messages)
        assert max_len > 1 or max_len == -1, "max_len must be -1 (no limit) or > 1"
        assert bool(system_prompt) + bool(messages) <= 1
        self.fixed_n = fixed_n
        self.max_len = max_len
        if system_prompt:
            self.update(prompt=system_prompt, role=Role.SYSTEM)
        
    def append(self, chat: dict):
        Role.validate(chat["role"])
        if len(self) == self.max_len:
            self.pop(self.fixed_n)    # i.e. keep 0, 1, ..., n-1 (first n)
        super().append(chat)

    def update(self, prompt: str, role: str):
        """Append a message to the chat history."""
        self.append(message_dict(prompt=prompt, role=role))


chat_history = ChatHistory(
    system_prompt="you are a goldfish", max_len=3, fixed_n=1
)
chat_history.update("1", "user")
chat_history.update("2", "user")
chat_history.update("3", "user")
chat_history

[{'role': 'system', 'content': 'you are a goldfish'},
 {'role': 'user', 'content': '2'},
 {'role': 'user', 'content': '3'}]

In [123]:
for cmd in [
    lambda: chat_history.append({"role": "test", "content": "test"}),
    lambda: chat_history.update(role="test", prompt="test")
]:
    try:
        cmd()
    except ValueError as e:
        print(e)

Invalid role: test
Invalid role: test


### Tag extraction

The following utilities will be used to extract content from tags (e.g. `<thought>`, `<response>`, etc).

In [ ]:
import re
from dataclasses import dataclass


@dataclass
class TagContentResult:
    content: list[str]
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """
    Extracts all content enclosed by specified tags, 
    e.g. <thought>, <response>, etc.
    Parameters:
        text (str): The input string containing multiple potential tags
        tag  (str): The name of the tag to search for
    """
    tag_pattern = rf"<{tag}>(.*?)</{tag}>"
    matched_contents = re.findall(tag_pattern, text, re.DOTALL)

    return TagContentResult(
        content=[content.strip() for content in matched_contents],
        found=bool(matched_contents),
    )

message = """
<thought>This is a thought.</thought> 
<response>This is a response.</response>
<thought>This is another thought.</thought> 
"""
print(extract_tag_content(message, "thought"))
print(extract_tag_content(message, "response"))
print(extract_tag_content(message, "tool"))

TagContentResult(content=['This is a thought.', 'This is another thought.'], found=True)
TagContentResult(content=['This is a response.'], found=True)
TagContentResult(content=[], found=False)


## Pattern 1: Reflection

Reflection makes the agent reflect on its output. Or more generally, it makes multiple LLMs talk to each other in something like a **peer review process**. The reflection agent suggests modifications, additions, improvements in the writing style, and so on. This iterative process often leads to substantial gains in output quality, as the generation model benefits from external critique (i.e. different model, or just a different execution process[^reflection]).

[^reflection]: The reflection model is focused on evaluation, error detection, factual verification, or alignment with constraints. So it has a more concrete goal than generating content from scratch.

![**Reflection Pattern**. Two LLMs iteratively improve the generated response through an iterative review process.](./img/pattern-reflective.png){#fig-pattern-reflective}

**Setting up.** We will be using the [Groq service](https://console.groq.com/docs/overview) which, as of writing, has a generous free-tier offering for a limited number of models.

In [4]:
import pandas as pd

from openai import OpenAI
from notebooks.utils import load_dotenv, print
from IPython.display import display_markdown

load_dotenv(verbose=True)

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


In [5]:
client = OpenAI()

### System prompts

We will create two separate chat histories, one for generation and another for reflection. We set the generation system prompt as a developer tasked to write high-quality Python code. On the other hand, we set the reflection system prompt such that it only responds with feedback instead of rewriting the whole thing. Finally, we instruct the reflection agent to write `APPROVED` when satisfied so we can terminate the loop.

In [81]:
STOP_WORD = "APPROVED"

BASE_GENERATION_SYSTEM_PROMPT = r"""
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

BASE_REFLECTION_SYSTEM_PROMPT = rf"""
You are tasked with generating critique and recommendations on the user's generated content. 
Your role is to help the user improve by pointing out strengths, weaknesses, and opportunities 
for refinement. 

You must NEVER provide full solutions, rewritten versions of the content, or long verbatim outputs. 
You may use short illustrative examples (1-3 lines or a single sentence) only when necessary to clarify 
a point. Providing a complete solution is a policy violation. 

If the user content has something wrong or something to be improved, output ONLY a clear list of 
recommendations and critiques. 

If you are satisfied and have no further strong recommendations, output EXACTLY the single word:

{STOP_WORD}

GUIDELINES FOR CRITIQUE:
- Forbidden Example: Rewriting the entire essay, code, or design for the user.
- Forbidden Example: Giving the full, corrected version of the user's work.
- Allowed Example: "Consider clarifying your thesis statement, e.g., make it one clear sentence."
- Good Example: Pointing out issues, suggesting improvements, or giving high-level recommendations without completing the work for the user.

GUIDLINES FOR APPROVAL:
- You must be fully satisfied with the content before approving.
- You must have checked that all past issues have been fully addressed.
- You must be sure there are no remaining issues, weaknesses, or areas for improvement.
- "{STOP_WORD}" must appear alone on a line, with no emojis, punctuation, or explanations.
- Do not mix "{STOP_WORD}" with any feedback or comments.
- Forbidden Example: "{STOP_WORD}, but consider improving your introduction."
- Good Example: "{STOP_WORD}"
"""

SHARED_DEFINITION_OF_DONE = """
DEFINITION OF DONE: The best solution is the SIMPLEST correct implementation that:
- Solves the problem completely
- Is readable and maintainable  
- Avoids unnecessary complexity or over-engineering
- Uses appropriate level of robustness (not maximally robust)
- Prioritizes clarity over cleverness
"""

CODE_GENERATION_SYSTEM_PROMPT = "\n\n".join(["""
You are a Python programmer tasked with generating high quality Python code.
Generate exactly one Python implementation that prioritizes SIMPLICITY, READABILITY, and PRACTICALITY.
Aim for the simplest correct solution that solves the problem without over-engineering.
Avoid unnecessary complexity, clever tricks, or advanced features unless absolutely necessary.
Do not provide multiple options, explanations, or alternative approaches.
Output only the final code in a fenced Python block.
""", SHARED_DEFINITION_OF_DONE, BASE_GENERATION_SYSTEM_PROMPT])

CODE_REFLECTION_SYSTEM_PROMPT = "\n\n".join(["""
You are a Python programmer and strict code reviewer. 
Your goal is to produce the simplest correct solution possible.
Avoid unnecessary complexity, clever tricks, or over-engineering.
Prioritize readability, maintainability, and clarity over novelty.

Providing a complete solution is a policy violation. 
Forbidden Example (DO NOT DO THIS): Providing a full class or function rewrite. 
Your role is to help the user learn by giving feedback, not by coding for them. 
Allowed Example: “Consider validating input type, e.g., `if not isinstance(n, int): ...` ”

FORMAT REQUIREMENT:
- You MUST format your feedback in a **Markdown table** with two columns:
  | Issue | Details | Recommendation |
- Each row should contain exactly one critique and its corresponding recommendation.
- Do not use bullet points, numbered lists, or plain text for critiques — only a Markdown table.
""", SHARED_DEFINITION_OF_DONE, BASE_GENERATION_SYSTEM_PROMPT])

generation_chat_history = ChatHistory()
generation_chat_history.update(prompt=CODE_GENERATION_SYSTEM_PROMPT, role="system")

reflection_chat_history = ChatHistory()
reflection_chat_history.update(prompt=CODE_REFLECTION_SYSTEM_PROMPT, role="system")

:::{.callout-caution}
Tuning the prompts took the most time / effort during the writing of this section. (ᵕ—ᴗ—) What worked for me: adding a shared **definition of done**, and having similar goals for both agents. In theory, having divergent goals can be good, but in practice it lead to agents going off-track, or getting into add-remove cycles. Or one agent dominating the other.
:::

### Model choice

Next, we choose the LLM models:

In [82]:
GENERATION_MODEL = "gpt-4.1-mini"
REFLECTION_MODEL = "o3"

For the current task (generating code for a simple function), we found:

| Role        | Focus                                   | Example size | Reasoning demand |
|-------------|------------------------------------------------------|--------------|------------------|
| **Generation** | Creativity, fluency, diverse output                  | ~8B         | Moderate         |
| **Reflection** | Evaluation, error detection, factual verification, constraint alignment | ≥20B | High             |


From our experiments (and performing code review IRL), reviewing is a nontrivial task: following guidelines, spotting subtle issues, and enforcing consistency needs strong reasoning capacity and attention to detail. We generally had best results with a smaller generation model paired with a larger [reasoning model]{.underline} (e.g. `o4` and `gpt-oss`) as reflection model.

### Generation step

We now ask the LLM to write an implementation of the Fibonacci sequence. Since it's only used for a quick demo, we expect the agents to converge to a simple solution.

In [83]:
# push user prompt to both agents => shared goal
USER_PROMPT = """
Generate a Python implementation of merge sort. This will only be used for a quick demo.
"""
generation_chat_history.update(role="user", prompt=USER_PROMPT)
reflection_chat_history.update(role="user", prompt=USER_PROMPT)

**Initial version.** As usual, GA has role `assistant`. We send over the response to the RA with role `user`:

In [84]:
completion = client.chat.completions.create(
    messages=generation_chat_history,
    model=GENERATION_MODEL
)

code = completion.choices[0].message.content
generation_chat_history.update(prompt=code, role="assistant")
reflection_chat_history.update(prompt=code, role="user")

:::{.callout-note collapse="true"}
## Initial generated code

In [66]:
#| echo: false
display_markdown(code, raw=True)

```python
def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    return merge(left, right)

def merge(left, right):
    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged

# Demo
if __name__ == "__main__":
    example = [38, 27, 43, 3, 9, 82, 10]
    print("Original:", example)
    sorted_example = merge_sort(example)
    print("Sorted:", sorted_example)
```

:::

### Reflection step

Generating the RA feedback. The generated critique is likewise sent over to the GA.

In [67]:
completion = client.chat.completions.create(
    messages=reflection_chat_history,
    model=REFLECTION_MODEL
)

critique = completion.choices[0].message.content
reflection_chat_history.update(prompt=critique, role="assistant")
generation_chat_history.update(prompt=critique, role="user")

:::{.callout-note collapse="true"}
## Feedback from reflection

In [68]:
#| echo: false
display_markdown(critique, raw=True)

| Issue | Details | Recommendation |
|-------|---------|----------------|
| Lack of documentation | Neither function explains its purpose, expected input, or return value. | Add concise docstrings describing parameters, algorithm behavior (stable, not-in-place), and complexity. |
| Unnecessary list slicing | `left = merge_sort(arr[:mid])` and `right = merge_sort(arr[mid:])` copy data at every recursive call, increasing time and memory use to Θ(n log n) extra space. | Consider passing start/end indices or using iterators to avoid repeated copying if the demo ever handles large inputs. |
| Potential recursion limit issues | Deep recursion on very large lists can raise `RecursionError` due to Python’s default recursion depth (~1000). | Guard against this with an explicit size check or convert to an iterative, bottom-up merge sort when scalability matters. |
| No input validation | Functions assume `arr` is an iterable of mutually comparable elements; unexpected types will raise runtime errors. | Validate input (e.g., `isinstance(arr, list)`) or document that only sortable lists are supported. |
| Missing unit tests | Only a single hard-coded demo list is exercised, so edge cases like empty lists, duplicates, or already-sorted input aren’t verified. | Add simple assertions or use `unittest`/`pytest` to cover common cases and ensure future changes don’t introduce regressions. |

:::

:::{.callout-tip}
Out of all models we've tested, only `o4-mini` strictly followed the review format.

:::

**Histories.** Chat histories after the first exchange. This will be followed by the second generation step, and we keep iterating until the stopping condition is triggered by the RA.

In [69]:
pd.DataFrame(generation_chat_history)

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,```python\ndef merge_sort(arr):\n if len(ar...
3,user,| Issue | Details | Recommendation |\n|-------...


In [70]:
pd.DataFrame(reflection_chat_history)

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,\nGenerate a Python implementation of merge so...
2,user,```python\ndef merge_sort(arr):\n if len(ar...
3,assistant,| Issue | Details | Recommendation |\n|-------...


### Full implementation

In [71]:
class ReflectionAgent:
    def __init__(self, 
        client, 
        generation_model: str, 
        reflection_model: str,
        generation_system_prompt: str = "",
        reflection_system_prompt: str = "",
    ):
        self.client = client
        self.generation_model = generation_model
        self.reflection_model = reflection_model
        self.generation_system_prompt = generation_system_prompt + BASE_GENERATION_SYSTEM_PROMPT
        self.reflection_system_prompt = reflection_system_prompt + BASE_REFLECTION_SYSTEM_PROMPT


    def _request_completion(self,
        history: list,
        model: str,
        verbose: int = 0,
        log_title: str = "COMPLETION",
    ) -> str:
        """Return completion content from the Groq model."""
        output = completions_create(self.client, history, model)
        if verbose > 0:
            print(f"\n\n{log_title}\n\n", output)

        return output

    def generate(self, generation_history: list, verbose: int = 0) -> str:
        """Generates response based on generation history using the generation model."""
        return self._request_completion(
            generation_history, self.generation_model,
            verbose, log_title="GENERATION"
        )

    def reflect(self, reflection_history: list, verbose: int = 0) -> str:
        """Generates feedback based on reflection history using the reflection model."""
        return self._request_completion(
            reflection_history, self.reflection_model,
            verbose, log_title="REFLECTION"
        )

    def run(self,
        user_prompt: str,
        max_iter: int = 10,
        verbose: int = 0,
        history_max_len: int = 5,
        generation_fixed_n: int = 1,   # e.g. 1 keep system prompt, 2 additionally keep user prompt
        reflection_fixed_n: int = 1,
    ) -> str:
        """
        Trigger the generation-reflection cycles over multiple steps based
        on a user prompt until max_iter or `APPROVED` is found in the feedback.
        Return (str) the final generated response after all cycles are completed.
        """
        
        generation_history = ChatHistory(max_len=history_max_len, fixed_n=generation_fixed_n)
        generation_history.update(prompt=self.generation_system_prompt, role="system")
        generation_history.update(prompt=user_prompt, role="user")

        reflection_history = ChatHistory(max_len=history_max_len, fixed_n=reflection_fixed_n)
        reflection_history.update(prompt=self.reflection_system_prompt, role="system")
        reflection_history.update(prompt=user_prompt, role="user")
        
        if verbose > 0:
            print("\n\nUSER\n\n", user_prompt)

        for step in range(max_iter):
            if verbose > 0:
                print("\n" + "=" * 80)
                print(f"Step [{step + 1}/{max_iter}]")
                print("=" * 80 + "\n")

            # Generate the response. Push to reflection as user
            generation = self.generate(generation_history, verbose=verbose)
            generation_history.update(prompt=generation, role="assistant")
            reflection_history.update(prompt=generation, role="user")

            # Critique the generation. Push to generation as user
            critique = self.reflect(reflection_history, verbose=verbose)
            reflection_history.update(prompt=critique, role="assistant")
            generation_history.update(prompt=critique, role="user")

            if STOP_WORD in critique:
                print("\n\n[Stop Sequence found. Stopping the reflection loop.]")
                break
            
        return {
            "generation": generation,
            "steps": step + 1,
            "generation_history": generation_history,
            "reflection_history": reflection_history,
        }

Running the process for a few iterations:

In [72]:
reflection_agent = ReflectionAgent(
    client=client,
    generation_model=GENERATION_MODEL,
    reflection_model=REFLECTION_MODEL,
    generation_system_prompt=CODE_GENERATION_SYSTEM_PROMPT,
    reflection_system_prompt=CODE_REFLECTION_SYSTEM_PROMPT
)

output = reflection_agent.run(
    user_prompt=USER_PROMPT,
    max_iter=10,
    verbose=1,
    history_max_len=5,
    generation_fixed_n=3,
    reflection_fixed_n=2 
)

# keep system, user prompt, first gen + last two generations 
# i.e. last generation-reflection pair which the GA can use to respond to improve



USER

 
Generate a Python implementation of merge sort. This will only be used for a quick demo.


Step [1/10]



GENERATION

 ```python
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

# Demo
if __name__ == "__main__":
    sample = [38, 27, 43, 3, 9, 82, 10]
    print("Original:", sample)
    print("Sorted:", merge_sort(sample))
```


REFLECTION

 | Issue | Details | Recommendation |
| --- | --- | --- |
| Missing documentation | The functions lack docstrings explaining purpose, parameters, return values, and complexity. | Add c

Comparing the results to see the effect of reflection:

:::{.callout-note collapse="true"}
## Initial output

In [79]:
#| echo: false
display_markdown(output["generation_history"][2]["content"], raw=True)

```python
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

# Demo
if __name__ == "__main__":
    sample = [38, 27, 43, 3, 9, 82, 10]
    print("Original:", sample)
    print("Sorted:", merge_sort(sample))
```

:::

:::{.callout-note collapse="true"}
## Approved output

In [80]:
#| echo: false
display_markdown(output["generation"], raw=True)

```python
from typing import TypeVar, Protocol, runtime_checkable, Sequence, Union

@runtime_checkable
class SupportsLessThan(Protocol):
    def __lt__(self, __other: object) -> bool: ...

T = TypeVar("T", bound=SupportsLessThan)

def merge_sort(arr: Union[list[T], tuple[T, ...]]) -> list[T]:
    """
    Return a new sorted list using merge sort.

    Args:
        arr: A list or tuple of comparable items supporting < operator.
             Assumes O(1) random access; other sequences might degrade performance.

    Returns:
        A new sorted list containing the elements of `arr`.

    Notes:
        - Uses index bounds instead of slicing to avoid some allocations,
          but still allocates new lists for merged results and recursion.
          This implementation has O(n log n) auxiliary space.
        - Stable sort: equal elements retain original order.
        - The helper function `_merge_sort_recursive` is a private top-level helper
          to simplify unit testing and profiling.
    """
    return _merge_sort_recursive(arr, 0, len(arr))


def _merge_sort_recursive(arr: Union[list[T], tuple[T, ...]], start: int, end: int) -> list[T]:
    if end - start <= 1:
        return [arr[start]] if end - start == 1 else []

    mid = (start + end) // 2
    left = _merge_sort_recursive(arr, start, mid)
    right = _merge_sort_recursive(arr, mid, end)
    return merge(left, right)


def merge(left: list[T], right: list[T]) -> list[T]:
    """
    Merge two sorted lists into one sorted list, preserving stability.
    """
    result: list[T] = []
    i = j = 0
    while i < len(left) and j < len(right):
        # Use 'not right[j] < left[i]' to treat <= without explicit '==', preserving stability
        if not right[j] < left[i]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result


def run_tests() -> None:
    """
    Basic tests verifying correctness of merge_sort.

    Raises AssertionError on failure.
    """
    test_cases = [
        ([], []),
        ([1], [1]),
        ([2, 1], [1, 2]),
        ([3, 1, 2, 1], [1, 1, 2, 3]),
        ([1, 2, 3, 4], [1, 2, 3, 4]),
        ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
        ([38, 27, 43, 3, 9, 82, 10], [3, 9, 10, 27, 38, 43, 82]),
    ]

    for i, (input_list, expected) in enumerate(test_cases, 1):
        output = merge_sort(input_list)
        assert output == expected, f"Test #{i} failed for input {input_list}: expected {expected}, got {output}"

    # Exception tests expect TypeError explicitly.
    try:
        merge_sort(123)  # Not list or tuple
    except TypeError:
        pass
    else:
        raise AssertionError("TypeError not raised for non-list/tuple input")

    try:
        merge_sort([1, object()])  # Elements not comparable
    except TypeError:
        pass
    else:
        raise AssertionError("TypeError not raised for non-comparable elements")

    print("All tests passed!")


if __name__ == "__main__":
    run_tests()
```

:::